In [1]:
from transformers import pipeline
import torch

model_id = "facebook/bart-large-mnli"

nli = pipeline(
    "text-classification",
    model=model_id,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    batch_size=12,
    torch_dtype="auto"
)

d:\PROJECTS\ProjectVision\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vivek\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
d:\PROJECTS\ProjectVision\.venv\Lib\site-packag

In [43]:
from typing_extensions import Any
import json
def verify_claim(claim: str, evidence: str)->dict[str, Any]:
    probs = nli(
        {
            "text": evidence,
            "text_pair": claim 
        },
        top_k=None
    )
    scores   = {
        item['label'].upper() : item['score']
    for item in probs
    }
    predicted_label = max(scores, key=scores.get)
    
    return {
        "claim": claim,
        "evidence": evidence,
        "predicted_label": predicted_label,
        "scores": scores
    }

def validate_claim(claim, evidence, threshold=0.85):

    result = verify_claim(claim, evidence)

    label = result.get("predicted_label")
    confidence = result["scores"][label]

    if confidence < threshold:
        status = "REVIEW_REQUIRED"

    elif label == "ENTAILMENT":
        status = "SUPPORTED"

    elif label == "CONTRADICTION":
        status = "CONTRADICTED"

    else:
        status = "INSUFFICIENT_EVIDENCE"

    return {
        **result,
        "status": status
    }

In [20]:
evidence = "Python was first released in 1991."

claim = "Python is widely used for AIML."

res = validate_claim(claim, evidence)
print(res)

{'claim': 'Python is widely used for AIML.', 'predicted_label': 'neutral', 'scores': {'neutral': 0.9996446371078491, 'entailment': 0.00022311796783469617, 'contradiction': 0.0001322390598943457}, 'status': 'INSUFFICIENT_EVIDENCE'}


In [44]:
claims = [
    "Python was first released in 1991.",
    "Python supports object-oriented programming.",
    "Python is exclusively used for web development."
]

documents = [
    {
        "source": "Python History",
        "content": "Python was first released in 1991."
    },
    {
        "source": "Python Documentation",
        "content": "Python supports multiple programming paradigms, including object-oriented programming."
    },
    {
        "source": "Python Overview",
        "content": "Python is used in web development, automation, data science, machine learning and other fields."
    }
]

for claim, evidence in zip(claims, documents):
    res = json.dumps(validate_claim(claim, evidence['content']), indent=4) 
    print(res)

{
    "claim": "Python was first released in 1991.",
    "evidence": "Python was first released in 1991.",
    "predicted_label": "ENTAILMENT",
    "scores": {
        "ENTAILMENT": 0.9866899251937866,
        "NEUTRAL": 0.012768727727234364,
        "CONTRADICTION": 0.0005413003382273018
    },
    "status": "SUPPORTED"
}
{
    "claim": "Python supports object-oriented programming.",
    "evidence": "Python supports multiple programming paradigms, including object-oriented programming.",
    "predicted_label": "ENTAILMENT",
    "scores": {
        "ENTAILMENT": 0.997450053691864,
        "NEUTRAL": 0.0023035542108118534,
        "CONTRADICTION": 0.0002463890705257654
    },
    "status": "SUPPORTED"
}
{
    "claim": "Python is exclusively used for web development.",
    "evidence": "Python is used in web development, automation, data science, machine learning and other fields.",
    "predicted_label": "CONTRADICTION",
    "scores": {
        "CONTRADICTION": 0.9900244474411011,
      